In [1]:

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    recall_score,
    f1_score
)
from sklearn.utils.class_weight import compute_sample_weight
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# 1. CARGAR Y PREPARAR LOS DATOS
# ============================================================

df = pd.read_csv(
    '/kaggle/input/datasets/nico14h/metagenomica/dataset.csv',
    sep='\t'
)

df_pivot = df.pivot(
    index='Paciente',
    columns='Bacteria_Limpia',
    values='AbundanciaTaxonomica'
).fillna(0)

labels = (
    df[['Paciente', 'Diagnosticos']]
    .drop_duplicates()
    .set_index('Paciente')
)

data = df_pivot.join(labels)

X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].astype(int).values

nombres_bacterias = X_df.columns

# TEA = 1 (CLASE POSITIVA DE INTERÉS)
# HC  = 0 (CLASE NEGATIVA)

assert set(np.unique(y)) == {0, 1}, \
    "Las clases deben ser TEA=1 y HC=0"

print(f"Total de pacientes: {len(y)}")
print(f"Pacientes TEA: {np.sum(y == 1)}")
print(f"Pacientes HC: {np.sum(y == 0)}")
print(f"Total de bacterias: {X.shape[1]}")


# ============================================================
# 2. DEFINIR LAS MÉTRICAS
# ============================================================

def calcular_metricas(y_real, y_pred, prob_tea):

    # TEA=1 es la clase positiva para ROC-AUC
    y_binario = (y_real == 1).astype(int)

    auc = roc_auc_score(y_binario, prob_tea)

    bal_acc = balanced_accuracy_score(y_real, y_pred)

    # Sensibilidad: proporción de TEA detectados
    sensibilidad = recall_score(
        y_real,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    # Especificidad: proporción de HC detectados
    especificidad = recall_score(
        y_real,
        y_pred,
        pos_label=0,
        zero_division=0
    )

    # F1-score para TEA
    f1 = f1_score(
        y_real,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    return {
        'ROC-AUC': auc,
        'Balanced Accuracy': bal_acc,
        'Sensibilidad (TEA)': sensibilidad,
        'Especificidad (HC)': especificidad,
        'F1-score (TEA)': f1
    }


# ============================================================
# 3. DEFINIR EL MODELO XGBOOST
# ============================================================

def crear_modelo():

    return XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        reg_lambda=2.0,
        importance_type='gain',
        #random_state=42,
        n_jobs=-1
    )


# ============================================================
# 4. VALIDACIÓN CRUZADA ESTRATIFICADA DE 5 FOLDS
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultados_folds = []

# Predicciones out-of-fold
predicciones_oof = np.empty(len(y), dtype=int)
probabilidades_oof = np.empty(len(y), dtype=float)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    modelo = crear_modelo()

    # Balanceo de clases usando únicamente el entrenamiento
    pesos = compute_sample_weight(
        class_weight='balanced',
        y=y_train
    )

    modelo.fit(
        X_train,
        y_train,
        sample_weight=pesos
    )

    # Predicciones
    y_pred = modelo.predict(X_test)

    # Probabilidad de TEA = clase 1
    indice_tea = np.where(modelo.classes_ == 1)[0][0]
    prob_tea = modelo.predict_proba(X_test)[:, indice_tea]

    # Guardar predicciones out-of-fold
    predicciones_oof[test_idx] = y_pred
    probabilidades_oof[test_idx] = prob_tea

    # Calcular métricas del fold
    metricas = calcular_metricas(
        y_test,
        y_pred,
        prob_tea
    )

    metricas['Fold'] = fold
    resultados_folds.append(metricas)


df_resultados = pd.DataFrame(
    resultados_folds
).set_index('Fold')


# ============================================================
# 5. MOSTRAR RESULTADOS POR FOLD
# ============================================================

print("\n========== XGBOOST: RESULTADOS POR FOLD ==========")

print(df_resultados.round(4).to_string())


# ============================================================
# 6. MEDIA Y DESVIACIÓN ESTÁNDAR
# ============================================================

resumen = pd.DataFrame({
    'Media': df_resultados.mean(),
    'Desviacion_Estandar': df_resultados.std(ddof=1)
})

print("\n========== XGBOOST: RESUMEN 5-FOLD CV ==========")

for metrica, fila in resumen.iterrows():

    print(
        f"{metrica:25s}: "
        f"{fila['Media']:.4f} ± "
        f"{fila['Desviacion_Estandar']:.4f}"
    )


# ============================================================
# 7. INTERVALOS DE CONFIANZA 95% - BOOTSTRAP OOF
# ============================================================

n_bootstrap = 2000
rng = np.random.default_rng(42)

indices_tea = np.where(y == 1)[0]
indices_hc = np.where(y == 0)[0]

# Métricas globales usando predicciones out-of-fold
metricas_oof = calcular_metricas(
    y,
    predicciones_oof,
    probabilidades_oof
)

valores_bootstrap = {
    metrica: [] for metrica in metricas_oof
}

for _ in range(n_bootstrap):

    # Remuestreo estratificado con reemplazo
    muestra_tea = rng.choice(
        indices_tea,
        size=len(indices_tea),
        replace=True
    )

    muestra_hc = rng.choice(
        indices_hc,
        size=len(indices_hc),
        replace=True
    )

    indices = np.concatenate([
        muestra_tea,
        muestra_hc
    ])

    metricas_boot = calcular_metricas(
        y[indices],
        predicciones_oof[indices],
        probabilidades_oof[indices]
    )

    for metrica, valor in metricas_boot.items():
        valores_bootstrap[metrica].append(valor)


print("\n========== XGBOOST: IC 95% (BOOTSTRAP OOF) ==========")

for metrica, valores in valores_bootstrap.items():

    ic_inf, ic_sup = np.percentile(
        valores,
        [2.5, 97.5]
    )

    print(
        f"{metrica:25s}: "
        f"{metricas_oof[metrica]:.4f} "
        f"[{ic_inf:.4f}, {ic_sup:.4f}]"
    )


# ============================================================
# 8. IMPORTANCIA DE LAS BACTERIAS
# ============================================================

# Modelo final entrenado con todos los pacientes
modelo_final = crear_modelo()

pesos_finales = compute_sample_weight(
    class_weight='balanced',
    y=y
)

modelo_final.fit(
    X,
    y,
    sample_weight=pesos_finales
)

importancias = modelo_final.feature_importances_

df_importancias = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Importancia_Predictiva': importancias
}).sort_values(
    by='Importancia_Predictiva',
    ascending=False
)


# ============================================================
# 9. PERFIL BIOLÓGICO TEA VS HC
# ============================================================

medias = data.groupby('Diagnosticos').mean().T

# TEA=1, HC=0
medias = medias.rename(columns={
    1: 'Media_TEA',
    0: 'Media_HC'
})

medias['Diferencia'] = (
    medias['Media_TEA'] - medias['Media_HC']
)

medias['Estado_en_TEA'] = np.select(
    [
        medias['Diferencia'] > 0,
        medias['Diferencia'] < 0
    ],
    [
        'Aumentada',
        'Disminuida'
    ],
    default='Sin cambio'
)

df_final = df_importancias.merge(
    medias[['Estado_en_TEA']],
    left_on='Bacteria',
    right_index=True
)

df_final = df_final[
    df_final['Importancia_Predictiva'] > 0
]

print("\n========== TOP 20 BACTERIAS XGBOOST ==========")

print(df_final.head(20).to_string(index=False))


# ============================================================
# 10. GUARDAR RESULTADOS
# ============================================================

resumen.to_csv('xgboost_metricas_cv5.csv')
df_resultados.to_csv('xgboost_resultados_folds.csv')
df_final.to_csv('xgboost_importancia_bacterias.csv', index=False)

pd.DataFrame({
    'Paciente': data.index,
    'Diagnostico_real': y,
    'Prediccion_OOF': predicciones_oof,
    'Probabilidad_TEA_OOF': probabilidades_oof
}).to_csv('xgboost_predicciones_oof.csv', index=False)

print("\nResultados exportados correctamente.")


Total de pacientes: 121
Pacientes TEA: 71
Pacientes HC: 50
Total de bacterias: 152

========== XGBOOST: RESULTADOS POR FOLD ==========
      ROC-AUC  Balanced Accuracy  Sensibilidad (TEA)  Especificidad (HC)  F1-score (TEA)
Fold                                                                                    
1      0.8333             0.7000              0.8000                 0.6          0.7742
2      0.8571             0.7786              0.8571                 0.7          0.8276
3      0.9857             0.8643              0.9286                 0.8          0.8966
4      0.9143             0.8786              0.8571                 0.9          0.8889
5      0.9643             0.8786              0.8571                 0.9          0.8889

========== XGBOOST: RESUMEN 5-FOLD CV ==========
ROC-AUC                  : 0.9110 ± 0.0659
Balanced Accuracy        : 0.8200 ± 0.0790
Sensibilidad (TEA)       : 0.8600 ± 0.0456
Especificidad (HC)       : 0.7800 ± 0.1304
F1-score (TEA)      